# Production VCF stats — GOLDEN STANDARD

**Source:** `pangenie_genotyping/data/merged/founders_231_chr.vcf.gz`

- 231 founder samples (80 cactus + 151 PanGenie short-read)
- 5,214,959 records (multi-allelic preserved)
- Mixed ploidy: cactus 80 haploid (`0`/`1`/`.`); PanGenie 151 diploid (`0/0`/`0/1`/`1/1`)
- **Beagle imputation explicitly NOT applied** — see `RESULTS_LOG.md` 2026-05-06 entry for literature backing.

Reads:
- `preprocess_qc/output/merged_stats/founders_231_by_size_class.tsv`
- `preprocess_qc/output/merged_stats/founders_231_by_sample.tsv`
- `preprocess_qc/output/merged_stats/founders_231_per_record.tsv.gz`

(Imputed-VCF stats `founders_231_IMPUTED_*` exist on disk but are NOT used.)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

STATS = Path('../output/merged_stats')
by_class  = pd.read_csv(STATS/'founders_231_by_size_class.tsv', sep='\t')
by_sample = pd.read_csv(STATS/'founders_231_by_sample.tsv',     sep='\t')
per_record= pd.read_csv(STATS/'founders_231_per_record.tsv.gz', sep='\t')

CLASS_ORDER = ['SNP','small_indel','small_sv','medium_sv','large_sv']
by_class = by_class.set_index('size_class').reindex(CLASS_ORDER)
print(f'Loaded {len(per_record):,} records, {len(by_sample)} samples')
print(f'Cohort split: cactus 80, pangenie 151')
by_class[['n_records','ac_all','an_all','missing_all','records_any_missing']].astype(int)

## 1. Catalog composition by size class

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
ax = axes[0]
by_class['n_records'].plot(kind='bar', ax=ax, color='#1f4e79', edgecolor='k')
ax.set_yscale('log'); ax.set_ylabel('# records (log)')
ax.set_title('Records per size class — production catalog')
for i, n in enumerate(by_class['n_records']):
    ax.text(i, n*1.1, f'{int(n):,}', ha='center', va='bottom', fontsize=9, rotation=0)
ax.tick_params(axis='x', rotation=15)

ax = axes[1]
miss_pct = by_class['records_any_missing'] / by_class['n_records'] * 100
miss_pct.plot(kind='bar', ax=ax, color='#d8b365', edgecolor='k')
ax.set_ylabel('% records with any missing')
ax.set_title('% records with ≥1 missing GT (mostly cactus haploid `.`)')
for i, p in enumerate(miss_pct):
    ax.text(i, p, f'{p:.1f}%', ha='center', va='bottom', fontsize=9)
ax.tick_params(axis='x', rotation=15)
plt.tight_layout(); plt.show()

## 2. MAF / MAC distribution by size class

In [ ]:
df = per_record.copy()
df['mac'] = np.minimum(df.ac_all, df.an_all - df.ac_all)
df['maf'] = df.mac / df.an_all.replace(0, 1)

thresholds = [1, 2, 5, 10]
rows = []
for cls in CLASS_ORDER:
    sub = df[df.size_class == cls]
    n = len(sub)
    row = {'size_class': cls, 'n_total': n}
    for t in thresholds:
        row[f'MAC>={t}']    = f'{(sub.mac >= t).sum():,} ({(sub.mac >= t).mean()*100:.1f}%)'
    for f in [0.01, 0.05, 0.10]:
        row[f'MAF>={f}']    = f'{(sub.maf >= f).sum():,} ({(sub.maf >= f).mean()*100:.1f}%)'
    rows.append(row)
summary = pd.DataFrame(rows).set_index('size_class')
summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, key, title, vals in [
    (axes[0], 'MAC', 'survival under MAC filter', [1,2,5,10]),
    (axes[1], 'MAF', 'survival under MAF filter', [0.01, 0.05, 0.10]),
]:
    width = 0.18
    x = np.arange(len(CLASS_ORDER))
    for i, t in enumerate(vals):
        if key == 'MAC':
            pcts = [(df[df.size_class==c].mac >= t).mean()*100 for c in CLASS_ORDER]
        else:
            pcts = [(df[df.size_class==c].maf >= t).mean()*100 for c in CLASS_ORDER]
        ax.bar(x + (i - len(vals)/2 + 0.5)*width, pcts, width, label=f'{key}>={t}')
    ax.set_xticks(x); ax.set_xticklabels(CLASS_ORDER, rotation=15)
    ax.set_ylabel('% records kept')
    ax.set_title(title)
    ax.legend(loc='lower left', fontsize=9)
    ax.set_ylim(0, 105)
plt.tight_layout(); plt.show()

## 3. Per-sample carrier counts (PanGenie 151 vs cactus 80)

In [ ]:
by_sample['n_sv_alt_total'] = by_sample[['n_small_sv_alt','n_medium_sv_alt','n_large_sv_alt']].sum(axis=1)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, col, title in zip(
    axes,
    ['n_alt', 'n_sv_alt_total'],
    ['# carrier records per sample (any size class)', '# SV-carrier records per sample (≥50bp)']):
    for cohort, color in [('cactus', '#7f8da6'), ('pangenie', '#1f4e79')]:
        sub = by_sample[by_sample.cohort == cohort]
        ax.hist(sub[col], bins=30, alpha=0.7, label=f'{cohort} (n={len(sub)})', color=color)
    ax.set_xlabel(col); ax.set_ylabel('# samples')
    ax.set_title(title); ax.legend()
plt.tight_layout(); plt.show()

for cohort in ['cactus','pangenie']:
    sub = by_sample[by_sample.cohort == cohort]
    print(f'  {cohort}  (n={len(sub)})')
    print(f'    SV carriers per sample:  median {int(sub.n_sv_alt_total.median()):,}  '
          f'IQR [{int(sub.n_sv_alt_total.quantile(0.25)):,}, {int(sub.n_sv_alt_total.quantile(0.75)):,}]')
    print(f'    Total carriers:          median {int(sub.n_alt.median()):,}')

## 4. Per-sample missingness

In [ ]:
by_sample['missing_pct'] = by_sample.n_missing / (by_sample.n_called + by_sample.n_missing) * 100
fig, ax = plt.subplots(figsize=(10, 4))
for cohort, color in [('cactus','#7f8da6'), ('pangenie','#1f4e79')]:
    sub = by_sample[by_sample.cohort == cohort]
    ax.hist(sub.missing_pct, bins=30, alpha=0.7,
            label=f'{cohort} (n={len(sub)}; median {sub.missing_pct.median():.2f}%)',
            color=color)
ax.set_xlabel('missing GT % per sample')
ax.set_ylabel('# samples')
ax.set_title('Per-sample missingness — cactus haploid `.` vs PanGenie diploid `./. `')
ax.legend()
plt.tight_layout(); plt.show()

## 5. Records with high cohort-specific missingness

Where cactus or PanGenie loses many founders simultaneously.

In [ ]:
# Per-record cohort missingness fraction
df['miss_pct_cact'] = df.missing_cact / 80 * 100
df['miss_pct_pang'] = df.missing_pang / 151 * 100
df['miss_pct_all']  = df.missing_all / 231 * 100

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist([df[df.size_class==c].miss_pct_cact for c in CLASS_ORDER],
             bins=20, label=CLASS_ORDER, stacked=False, alpha=0.7)
axes[0].set_yscale('log')
axes[0].set_xlabel('% cactus-side founders missing per record')
axes[0].set_ylabel('# records (log)')
axes[0].set_title('Cactus-side missingness per record')
axes[0].legend(loc='upper right', fontsize=8)

axes[1].hist([df[df.size_class==c].miss_pct_pang for c in CLASS_ORDER],
             bins=20, label=CLASS_ORDER, stacked=False, alpha=0.7)
axes[1].set_yscale('log')
axes[1].set_xlabel('% PanGenie-side founders missing per record')
axes[1].set_ylabel('# records (log)')
axes[1].set_title('PanGenie-side missingness per record')
axes[1].legend(loc='upper right', fontsize=8)
plt.tight_layout(); plt.show()

print(f'  records with >10% cactus-side missing:    {(df.miss_pct_cact > 10).sum():,} ({(df.miss_pct_cact > 10).mean()*100:.1f}%)')
print(f'  records with >10% PanGenie-side missing:  {(df.miss_pct_pang > 10).sum():,} ({(df.miss_pct_pang > 10).mean()*100:.1f}%)')
print(f'  records with >10% overall missing:        {(df.miss_pct_all  > 10).sum():,} ({(df.miss_pct_all  > 10).mean()*100:.1f}%)')

## 6. Headline summary

In [ ]:
n_total = by_class.n_records.sum()
miss_total = by_class.missing_all.sum()
an_total = by_class.an_all.sum()
ac_total = by_class.ac_all.sum()

print(f'== GOLDEN-STANDARD VCF (founders_231_chr.vcf.gz) ==')
print(f'  samples: 231 (80 cactus + 151 PanGenie)')
print(f'  records: {int(n_total):,}')
print(f'  total GT cells:   {int(an_total) + int(miss_total):,}')
print(f'  called cells:     {int(an_total):,}  ({an_total/(an_total+miss_total)*100:.2f}%)')
print(f'  missing cells:    {int(miss_total):,}  ({miss_total/(an_total+miss_total)*100:.2f}%)')
print(f'  total alt-allele observations:  {int(ac_total):,}')
print()
print(f'  records with any missing call: {int(by_class.records_any_missing.sum()):,} ({by_class.records_any_missing.sum()/n_total*100:.1f}%)')
print(f'  most missingness is cactus haploid `.` (informative absence, NOT quality-driven missingness)')